
# explore_external_sources.ipynb

**Purpose**: Explore and map **external data sources** to power **Spec 1.1 — External Data Extraction & Ingestion**.  
This notebook provides:
- Live API-call scaffolding (FRED, World Bank, LME-style feed, USGS XLS)  
- Mock fallbacks so you can test transformations without internet  
- Validation checks and basic exploratory plots  
- Output CSVs to `/data/raw` and `/data/processed` and a metadata YAML stub

> Replace placeholder keys/URLs with real credentials and endpoints in your environment.


In [ ]:

# --- 1) Setup & Configuration ---
import os
import io
import sys
import json
import time
import yaml
import math
import zipfile
import warnings
import datetime as dt
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings('ignore')

# Project dirs (adjust paths for your machine/repo layout)
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'
META_DIR = BASE_DIR / 'config'

for d in [DATA_DIR, RAW_DIR, PROC_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# API Keys (set in your shell before running live calls)
FRED_API_KEY = os.getenv('FRED_API_KEY', 'YOUR_FRED_KEY_HERE')  # replace in env
# LME and other providers may require auth tokens; place here if applicable
LME_API_TOKEN = os.getenv('LME_API_TOKEN', 'YOUR_LME_TOKEN_HERE')

print('Directories ready:', DATA_DIR, RAW_DIR, PROC_DIR, META_DIR, sep='\n')


In [ ]:

# --- 2) Helper functions ---
def fetch_json(url, params=None, headers=None, timeout=30):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"[WARN] JSON fetch failed: {e}")
        return None

def fetch_csv(url, params=None, headers=None, timeout=30):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        return pd.read_csv(io.StringIO(r.text))
    except Exception as e:
        print(f"[WARN] CSV fetch failed: {e}")
        return None

def to_date(s):
    try:
        return pd.to_datetime(s).date()
    except Exception:
        return pd.NaT

def write_df(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"[OK] Wrote {len(df):,} rows -> {path}")

def plot_series(df, x_col, y_col, title):
    plt.figure()
    plt.plot(pd.to_datetime(df[x_col]), df[y_col])
    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.show()



## FRED: CPI / Industrial Production / Rates (Live + Mock)

This cell pulls CPI from **FRED** using `FRED_API_KEY`. If the call fails (no internet or key), it falls back to a mock series.


In [ ]:

# --- 3) FRED CPI example ---
FRED_URL = 'https://api.stlouisfed.org/fred/series/observations'
series_id = 'CPIAUCSL'  # CPI All Urban Consumers, Index 1982-84=100

params = {
    'series_id': series_id,
    'api_key': FRED_API_KEY,
    'file_type': 'json',
    'observation_start': '2010-01-01'
}

fred_json = fetch_json(FRED_URL, params=params)

if fred_json and 'observations' in fred_json:
    df_cpi = pd.DataFrame(fred_json['observations'])[['date','value']]
    df_cpi['date'] = pd.to_datetime(df_cpi['date'])
    df_cpi['value'] = pd.to_numeric(df_cpi['value'], errors='coerce')
    df_cpi.dropna(inplace=True)
    df_cpi['series_id'] = series_id
    print('[OK] Live FRED CPI fetched.')
else:
    # Mock series (monthly dates, simple trend)
    dates = pd.date_range('2010-01-01', periods=180, freq='MS')
    vals = 220 + np.linspace(0, 60, len(dates)) + np.random.normal(0, 0.6, len(dates))
    df_cpi = pd.DataFrame({'date': dates, 'value': vals, 'series_id': series_id})
    print('[MOCK] Using generated CPI series.')

write_df(df_cpi, RAW_DIR / 'fred_cpi.csv')
plot_series(df_cpi, 'date', 'value', 'FRED CPI (live or mock)')



## World Bank: Pink Sheet (Commodities) (Live + Mock)

Attempts to fetch a metal commodity indicator via the World Bank API; falls back to a mock if unavailable.


In [ ]:

# --- 4) World Bank commodity example ---
# Note: Replace indicator with an actual commodity series id if needed.
WB_URL = 'https://api.worldbank.org/v2/country/WLD/indicator/CM.MKT.INDX.ZG?format=json'  # placeholder series
wb_json = fetch_json(WB_URL)

if wb_json and isinstance(wb_json, list) and len(wb_json) > 1 and isinstance(wb_json[1], list):
    rows = wb_json[1]
    df_wb = pd.DataFrame([{
        'date': r.get('date'),
        'value': r.get('value'),
        'indicator': r.get('indicator', {}).get('id', 'UNKNOWN')
    } for r in rows])
    df_wb.dropna(subset=['value'], inplace=True)
    df_wb['date'] = pd.to_datetime(df_wb['date'])
    df_wb['value'] = pd.to_numeric(df_wb['value'], errors='coerce')
    df_wb.dropna(inplace=True)
    print('[OK] Live World Bank series fetched.')
else:
    # Mock monthly copper price (USD/ton) style
    dates = pd.date_range('2015-01-01', periods=120, freq='MS')
    base = 6000 + 500*np.sin(np.linspace(0, 12*np.pi, len(dates)))
    noise = np.random.normal(0, 120, len(dates))
    df_wb = pd.DataFrame({
        'date': dates,
        'value': base + noise,
        'indicator': 'PINK_SHEET_COPPER_USD_TON'
    })
    print('[MOCK] Using generated Pink Sheet-like copper series.')

write_df(df_wb, RAW_DIR / 'worldbank_commodity_copper.csv')
plot_series(df_wb, 'date', 'value', 'World Bank Copper Proxy (live or mock)')



## LME/Exchange Daily Settlement (Mock Scaffold)

Real LME/COMEX endpoints often require paid access. Below we use a **schema-accurate mock** and show how to wire a bearer token.


In [ ]:

# --- 5) LME-style mock ---
# Example of authenticated GET if you have access:
# url = 'https://api.lme.com/marketdata/v1/aluminium/settlements'
# headers = {'Authorization': f'Bearer {LME_API_TOKEN}'}
# df_lme = fetch_csv(url, headers=headers)

# Mock daily series for ALUMINUM, COPPER, NICKEL, STEEL_INDEX
dates = pd.date_range('2022-01-01', periods=750, freq='B')
metals = ['ALUMINUM','COPPER','NICKEL','STEEL_INDEX']

records = []
rng = np.random.default_rng(42)
for m in metals:
    level = {'ALUMINUM': 2300, 'COPPER': 7800, 'NICKEL': 22000, 'STEEL_INDEX': 900}[m]
    series = level + np.cumsum(rng.normal(0, 5 if m!='NICKEL' else 25, len(dates)))
    for d, v in zip(dates, series):
        records.append({'date': d, 'metal': m, 'price_usd_per_ton': max(v, 10.0), 'src': 'LME_MOCK'})

df_lme_mock = pd.DataFrame(records)
write_df(df_lme_mock, RAW_DIR / 'lme_settlements_mock.csv')
df_lme_mock.head()



## USGS XLS Parsing (Mock)

Demonstrates how to parse an Excel file layout (USGS often provides monthly tables). We generate a mock workbook layout here.


In [ ]:

# --- 6) USGS XLS parsing demo (mock workbook) ---
import pandas as pd

mock_xls_path = RAW_DIR / 'usgs_mock.xlsx'
with pd.ExcelWriter(mock_xls_path) as xw:
    df_sheet = pd.DataFrame({
        'Month': pd.date_range('2020-01-01', periods=24, freq='MS'),
        'Aluminum_USD_Ton': 1900 + np.linspace(0, 200, 24) + np.random.normal(0, 20, 24),
        'Copper_USD_Ton': 6500 + np.linspace(0, 400, 24) + np.random.normal(0, 30, 24)
    })
    df_sheet.to_excel(xw, index=False, sheet_name='Prices')

print('[OK] Wrote mock USGS workbook:', mock_xls_path)

df_usgs = pd.read_excel(mock_xls_path, sheet_name='Prices')
df_usgs.rename(columns={'Month':'date'}, inplace=True)
df_usgs = df_usgs.melt(id_vars=['date'], var_name='series', value_name='value')
write_df(df_usgs, RAW_DIR / 'usgs_parsed_mock.csv')
df_usgs.head()



## Normalize, Validate, and Merge Samples

- Normalize to `USD per ton` and daily or monthly calendars
- Basic schema checks and outlier flags


In [ ]:

# --- 7) Normalize/validate/merge ---
def basic_validate_prices(df, date_col='date', value_col='price_usd_per_ton'):
    problems = []
    if df[date_col].isna().any():
        problems.append('Null dates found')
    if (df[value_col] <= 0).any():
        problems.append('Non-positive prices encountered')
    return problems

# Normalize FRED CPI monthly to month-end
df_cpi_norm = df_cpi.copy()
df_cpi_norm['date'] = pd.to_datetime(df_cpi_norm['date']).dt.to_period('M').dt.to_timestamp('M')
write_df(df_cpi_norm, PROC_DIR / 'fred_cpi_norm.csv')

# LME mock is already daily business days and USD/ton
issues = basic_validate_prices(df_lme_mock, 'date', 'price_usd_per_ton')
print('Validation issues (LME mock):', issues or 'None')

# Simple outlier flag by z-score per metal
df_lme_proc = df_lme_mock.copy()
df_lme_proc['z'] = df_lme_proc.groupby('metal')['price_usd_per_ton'].transform(lambda s: (s - s.rolling(60).mean())/s.rolling(60).std())
df_lme_proc['outlier_flag'] = (df_lme_proc['z'].abs() > 4).astype(int)
write_df(df_lme_proc.drop(columns=['z']), PROC_DIR / 'lme_settlements_proc.csv')

# Quick plot
for m in df_lme_proc['metal'].unique():
    sub = df_lme_proc[df_lme_proc['metal']==m]
    plot_series(sub, 'date', 'price_usd_per_ton', f'{m} mock price')



## Metadata YAML Stub (for `config/external_sources.yaml`)


In [ ]:

meta = {
    'metals': [
        {'name':'ALUMINUM','provider':'LME','url':'<your_lme_endpoint>','unit':'USD/ton','freq':'daily'},
        {'name':'COPPER','provider':'LME','url':'<your_lme_endpoint>','unit':'USD/ton','freq':'daily'},
        {'name':'NICKEL','provider':'LME','url':'<your_lme_endpoint>','unit':'USD/ton','freq':'daily'},
        {'name':'STEEL_INDEX','provider':'<index_provider>','url':'<endpoint>','unit':'USD/ton','freq':'daily'},
    ],
    'macro': [
        {'series_id':'CPIAUCSL','provider':'FRED','url':'https://api.stlouisfed.org/fred/series/observations','freq':'monthly'},
        {'series_id':'INDPRO','provider':'FRED','url':'https://api.stlouisfed.org/fred/series/observations','freq':'monthly'}
    ],
    'schedule': {'daily':'23:30Z'}
}
meta_path = META_DIR / 'external_sources.yaml'
with open(meta_path, 'w') as f:
    yaml.safe_dump(meta, f, sort_keys=False)
print('[OK] Wrote metadata YAML ->', meta_path)
print(meta)



## Next Steps
1. Replace placeholder keys and endpoints with production credentials.  
2. Run the notebook in an environment with internet to validate live calls.  
3. Promote working code into Spec 1.1 Python modules (`extract.py`, `transform.py`, `load.py`).  
4. Add additional sources (OECD/IMF/USGS official) and map their schemas.  
5. Wire into your scheduler (Airflow/Prefect) and add alerting for SLA and schema drift.
